# 05 - Final Research Interpretation

## Reliable Machine-Failure Prediction Under Imbalanced and Corrupted Sensor Data

This notebook brings together the main findings from the baseline modelling, robustness experiments, and explainability analysis.

The objective is not simply to determine whether a machine-failure prediction model can achieve strong performance on clean test data. Instead, the project investigates whether the selected prediction system remains reliable when realistic data-quality problems are introduced and whether its predictions can be meaningfully explained.

The analysis therefore addresses three connected questions:

1. **Can the selected model detect machine failures effectively under clean-data conditions?**
2. **How does its reliability change when sensor measurements become noisy, incomplete, or when labelled training data are limited?**
3. **Which input features influence the model's decisions, and why are some failures detected while others are missed?**

The selected prediction system is an unweighted Random Forest evaluated using a fixed decision threshold of **0.20**. The same model configuration, preprocessing strategy, data split, and threshold are maintained throughout the project to ensure that the different experiments remain directly comparable.

## 2. Clean-Data Performance

The baseline modelling stage established the reference performance of the selected Random Forest under clean test conditions.

The final system achieved:

- **Accuracy:** 97.27%
- **Precision:** 56.76%
- **Recall:** 82.35%
- **F1-score:** 67.20%
- **Average Precision:** 78.03%
- **Decision threshold:** 0.20

The corresponding confusion matrix was:

```text
[[1417   32]
 [   9   42]]

## 3. Robustness Under Imperfect Data

The robustness experiments evaluated whether the same selected Random Forest remained reliable when realistic data-quality problems were introduced.

Three conditions were investigated:

1. **Sensor measurement noise**
2. **Missing sensor measurements**
3. **Limited labelled training data**

The clean-data system remained fixed throughout the experiments, including the same model configuration, preprocessing logic, test set, and decision threshold of **0.20**.

### 3.1 Sensor Noise

As sensor noise increased, failure-detection performance generally deteriorated.

Under severe **30% sensor noise**, the model achieved approximately:

- **Precision:** 30.09%
- **Recall:** 66.67%
- **F1-score:** 41.46%
- **Average Precision:** 54.17%
- **Accuracy:** 93.60%

Compared with the clean reference recall of **82.35%**, recall fell to **66.67%**.

This shows that noisy measurements can substantially reduce the model's ability to detect real failures even though overall accuracy remains relatively high.

### 3.2 Missing Sensor Measurements

Missing sensor measurements produced an even stronger reduction in failure-detection capability.

At **30% missing numerical sensor measurements**, recall decreased from the clean reference value of **82.35%** to approximately **50.98%**.

Accuracy remained high at approximately **96.87%**, but this should not be interpreted as equally reliable failure detection. Because healthy machines dominate the dataset, the model can still achieve high overall accuracy while missing a large proportion of actual failures.

Among the severe robustness conditions examined, missing sensor measurements produced the largest reduction in failure recall.

### 3.3 Limited Training Data

The limited-training-data experiment evaluated how performance changed when the Random Forest was trained using only a fraction of the original labelled training set.

Average Precision improved as more labelled data became available:

- **20% training data:** 65.46%
- **40% training data:** 69.78%
- **60% training data:** 74.84%
- **80% training data:** 77.81%
- **100% training data:** 78.03%

At **20% training data**, recall was approximately **66.67%**, compared with **82.35%** when the full training set was used.

These results indicate that the quantity of labelled training data is particularly important for learning reliable failure patterns, especially because machine failures represent the minority class.

### 3.4 Overall Robustness Interpretation

The robustness experiments demonstrate that strong clean-data performance does not guarantee equally reliable behaviour under imperfect operating conditions.

The representative recall values were:

- **Clean data:** 82.35%
- **30% sensor noise:** 66.67%
- **30% missing measurements:** 50.98%
- **20% training data:** 66.67%

Missing sensor measurements had the strongest negative effect on failure recall among the severe conditions tested.

Overall, sensor quality, sensor availability, and the amount of labelled training data all influence the reliability of the machine-failure prediction system.

## 4. Explainability Findings

The explainability stage investigated which features influenced the selected Random Forest and why some machine failures were detected while others were missed.

### 4.1 Global Feature Importance

Impurity-based Random Forest feature importance identified **torque** as the most influential feature overall, accounting for approximately **32.45%** of the model's total feature importance.

Rotational speed and tool wear also contributed substantially, while temperature measurements had more moderate importance. The encoded machine-type variables contributed relatively little compared with the numerical operating measurements.

These results indicate which variables the Random Forest used most strongly across the dataset, but they do not establish physical causation.

### 4.2 Global SHAP Findings

SHAP analysis extended the feature-importance results by showing both the magnitude and direction of individual feature effects.

Rotational speed, torque, tool wear, and temperature measurements were among the strongest contributors to predicted failure risk, while machine-type variables generally had SHAP values concentrated close to zero.

Higher torque values often pushed predictions toward failure. Rotational speed also had a strong influence, while tool wear and temperature effects varied across observations.

The SHAP results therefore supported the earlier feature-importance analysis while providing a more detailed view of how individual feature values influenced model predictions.

### 4.3 Correctly Detected Failure

A correctly detected failed machine received a predicted failure probability of **0.94**.

For this machine, the strongest SHAP contributions were:

- **Rotational speed = 2737 rpm:** SHAP = **+0.5036**
- **Torque = 8.8:** SHAP = **+0.3625**

The remaining features contributed only weakly by comparison.

This shows that the high-confidence failure prediction was driven primarily by the rotational-speed and torque pattern learned by the Random Forest.

### 4.4 Missed Failure

A selected false-negative machine actually belonged to the failure class but received a predicted failure probability of only **0.1367**, below the fixed decision threshold of **0.20**.

The strongest SHAP contributions included:

- **Rotational speed = 1371 rpm:** SHAP = **+0.0910**
- **Air temperature = 303.6 K:** SHAP = **+0.0906**
- **Process temperature = 312.2 K:** SHAP = **−0.0684**
- **Tool wear = 112:** SHAP = **−0.0206**
- **Torque = 54.6:** SHAP = **+0.0143**

The false-negative case therefore contained conflicting model evidence. Some variables pushed the prediction toward failure, while others pushed it away from failure.

### 4.5 Explainability Interpretation

The comparison between the correctly detected and missed failure showed that **rotational speed and torque were the largest factors separating the two predictions**.

The correctly detected failure received much stronger positive SHAP contributions from these variables, while the missed failure showed weaker and more mixed evidence.

This helps explain why the model achieved strong but imperfect recall. Some real failures closely resemble the feature patterns learned by the Random Forest and are detected confidently, whereas others do not produce sufficiently strong model evidence to cross the fixed decision threshold.

The explainability findings therefore complement the performance and robustness results by showing how learned feature patterns contribute to both successful and unsuccessful failure detection.

All feature-importance and SHAP interpretations describe associations learned by the Random Forest and should not be interpreted as direct physical causes of machine failure.

## 5. Overall Research Interpretation

The results of this project show that reliable machine-failure prediction cannot be assessed using clean-data accuracy alone.

Under clean test conditions, the selected Random Forest achieved strong performance, including **82.35% recall** and **78.03% Average Precision**, while correctly detecting **42 of the 51 actual failures**. However, the robustness experiments showed that this performance degraded when realistic data-quality problems were introduced.

Sensor noise reduced failure-detection capability, with recall decreasing to **66.67%** under 30% noise. Missing sensor measurements had an even stronger effect, reducing recall to approximately **50.98%** at the 30% missing-data level. Limited labelled training data also reduced model reliability, with recall falling to approximately **66.67%** when only 20% of the original training data were available.

These results demonstrate that the apparent reliability of a predictive-maintenance model depends strongly on the quality and availability of sensor measurements and on the amount of labelled training data available during model development.

The experiments also highlight the limitations of relying on accuracy for highly imbalanced machine-failure data. Even under severe corruption, accuracy remained relatively high because healthy machines represented the large majority of observations. Recall, Average Precision, F1-score, and the confusion matrix therefore provided more meaningful information about actual failure-detection capability.

Explainability analysis provided additional insight into the model's behaviour. Torque, rotational speed, and tool wear were among the most influential variables in the Random Forest, while machine-type variables contributed relatively little.

SHAP analysis further showed that the model did not rely on a single feature independently. Instead, predictions were influenced by combinations of operating measurements. The correctly detected failure received a high probability of **0.94**, driven primarily by strong contributions from rotational speed and torque. In contrast, the selected missed failure received a probability of only **0.1367** because the evidence provided by its features was weaker and partly conflicting.

This helps explain why the model can perform strongly overall while still missing certain genuine failures. Failures that strongly resemble patterns learned during training may be detected confidently, whereas failures with less typical or conflicting feature combinations may remain below the decision threshold.

Taken together, the findings support three main conclusions:

1. **Strong clean-data performance does not guarantee robustness under realistic data imperfections.**
2. **Missing sensor information can be particularly damaging to failure-detection recall.**
3. **Explainability is valuable for understanding both successful and missed failure predictions, especially in an imbalanced predictive-maintenance setting.**

The project therefore demonstrates the importance of evaluating predictive-maintenance systems using a combination of clean-data performance, robustness testing, class-imbalance-aware metrics, and model explainability rather than relying on a single headline accuracy value.

## 6. Limitations and Future Work

Although the project provides a structured evaluation of machine-failure prediction under clean and imperfect data conditions, several limitations should be considered.

### 6.1 Limitations

First, the analysis was conducted using a single benchmark dataset. Therefore, the observed performance and robustness trends may not generalize directly to other machines, industrial systems, or operating environments.

Second, the robustness experiments simulated sensor noise and missing measurements artificially. These controlled perturbations are useful for testing model sensitivity, but real industrial sensor faults may be more complex, persistent, correlated, or dependent on operating conditions.

Third, the limited-training-data experiment used reduced subsets of the available labelled data. Because machine failures are rare, results from individual reduced subsets may vary depending on which failure examples are included.

Fourth, the selected Random Forest was evaluated using a fixed decision threshold of **0.20**. This threshold was intentionally kept constant to ensure fair comparison across experiments, but different maintenance applications may require different trade-offs between missed failures and false alarms.

Fifth, the explainability analysis describes how the Random Forest uses the available features, but it does not establish direct physical causation. SHAP values and impurity-based feature importance explain model behaviour rather than proving that a particular sensor measurement caused a machine failure.

Finally, the project focused mainly on one selected Random Forest system after baseline model comparison. More advanced modelling approaches may behave differently under corrupted or incomplete sensor data.

### 6.2 Future Work

Future work could extend this project in several directions.

One important extension would be to evaluate the system using additional industrial predictive-maintenance datasets to determine whether the observed robustness patterns generalize across different machines and operating environments.

More realistic sensor-fault simulations could also be investigated, including persistent sensor drift, sensor bias, complete sensor dropout, correlated measurement errors, and time-dependent degradation.

Missing-data robustness could be improved by comparing different imputation strategies or by developing models that can handle unavailable sensor measurements more directly.

The limited-training-data analysis could be repeated across multiple random stratified subsets to estimate variability and provide confidence intervals for the performance metrics.

Future studies could also compare the Random Forest with more advanced methods such as gradient-boosted trees, neural networks, temporal models, or ensemble approaches while applying the same robustness-testing framework.

Because missed failures are particularly important in predictive maintenance, further work could investigate cost-sensitive learning, probability calibration, threshold optimization, and uncertainty estimation to improve failure-detection reliability.

Finally, explainability could be extended beyond individual examples by systematically analysing correctly detected failures, false negatives, false positives, and healthy machines to identify recurring model behaviour across different prediction outcomes.

## 7. Conclusion

This project investigated machine-failure prediction from the perspective of reliability rather than clean-data performance alone.

The selected Random Forest achieved strong baseline performance on clean test data, including **82.35% recall**, **67.20% F1-score**, and **78.03% Average Precision** using a fixed decision threshold of **0.20**.

However, the robustness experiments demonstrated that performance deteriorated when realistic data-quality problems were introduced. Sensor noise reduced failure-detection capability, missing sensor measurements produced the strongest reduction in recall among the severe conditions tested, and limited labelled training data also weakened model performance.

These findings show that a model that performs well under ideal test conditions may still become substantially less reliable when deployed under imperfect operating conditions.

The explainability analysis provided further insight into the behaviour of the selected Random Forest. Torque, rotational speed, and tool wear were among the most influential features, while machine type contributed relatively little. SHAP analysis showed that individual predictions were driven by combinations of feature effects rather than by a single measurement alone.

The comparison between a correctly detected failure and a false-negative case demonstrated why some failures are identified confidently while others are missed. The correctly detected case showed strong failure-associated SHAP contributions, particularly from rotational speed and torque, whereas the missed failure produced weaker and partly conflicting evidence.

Overall, the project demonstrates that reliable predictive-maintenance modelling should combine:

1. **Class-imbalance-aware performance evaluation**
2. **Robustness testing under realistic data imperfections**
3. **Consistent decision-threshold evaluation**
4. **Model explainability**
5. **Analysis of both successful and missed failure predictions**

The main conclusion is that predictive-maintenance systems should not be judged by a single high accuracy value. Their reliability should instead be evaluated by how effectively they detect rare failures, how well they tolerate imperfect sensor data, and whether their predictions can be understood and critically examined.